In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm  # EfficientNetV2 ve CvT gibi modeller iÃƒÂ§in kritik
import os
import time
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# --- UYARILAR VE OPTÃ„Â°MÃ„Â°ZASYON Ã„Â°Ãƒâ€¡Ã„Â°N (Ã„Â°steÃ„Å¸e BaÃ„Å¸lÃ„Â±) ---
# AdamW ve LR Scheduler'larÃ„Â± iÃƒÂ§in ek modÃƒÂ¼l (CvT ve EfficientNetV2 iÃƒÂ§in ÃƒÂ¶nemlidir)
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau

# UyarÃ„Â±larÃ„Â± gizlemek iÃƒÂ§in
import warnings
warnings.filterwarnings("ignore")

# --- GLOBAL AYARLAR ---
# Random Seed (Tekrarlanabilir sonuÃƒÂ§lar iÃƒÂ§in)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# --- KONFÃ„Â°GÃƒÅ“RASYON (EfficientNetV2) ---

# EfficientNetV2 S (Small) Modeli (timm kodu: efficientnetv2_s)
MODEL_NAME = 'tf_efficientnetv2_m'

# Deney AdÃ„Â±
EXPERIMENT_NAME = "EfficientNetV2_Medium_Baseline_MediumLarge_Run1"

# Hiperparametreler
# BATCH_SIZE'Ã„Â± GPU belleÃ„Å¸inde yer aÃƒÂ§mak iÃƒÂ§in 64'ten 32'ye dÃƒÂ¼Ã…Å¸ÃƒÂ¼relim.
BATCH_SIZE = 24
EPOCHS = 50
LEARNING_RATE = 0.0003
NUM_CLASSES = 8
# Modelin kendi yapÃ„Â±sÃ„Â±nda Dropout olduÃ„Å¸undan, transfer ÃƒÂ¶Ã„Å¸renme yaparken bu deÃ„Å¸eri dÃƒÂ¼Ã…Å¸ÃƒÂ¼k tutmak yeterli.
DROPOUT_RATE = 0.2

# --- DOSYA YOLLARI ---
DATA_DIR = "../data/prepared-data"

# SonuÃƒÂ§larÃ„Â±n kaydedileceÃ„Å¸i yer
OUTPUT_DIR = f"../models/pytorch/{EXPERIMENT_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Cihaz KontrolÃƒÂ¼
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Cihaz: {DEVICE}")
print(f"Model: {MODEL_NAME}")
print(f"KayÃ„Â±t Yeri: {OUTPUT_DIR}")

In [ ]:
# ImageNet Normalize DeÃ„Å¸erleri
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Veri DÃƒÂ¶nÃƒÂ¼Ã…Å¸ÃƒÂ¼mleri
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
}

# Datasetleri OluÃ…Å¸tur
image_datasets = {x: datasets.ImageFolder(os.path.join(DATA_DIR, x), data_transforms[x])
                  for x in ['train', 'val', 'test']}

# DataLoaderlarÃ„Â± OluÃ…Å¸tur
dataloaders = {x: DataLoader(image_datasets[x], batch_size=BATCH_SIZE,
                             shuffle=(x=='train'), num_workers=4, pin_memory=True)
               for x in ['train', 'val', 'test']}

dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val', 'test']}
class_names = image_datasets['train'].classes

print(f"SÃ„Â±nÃ„Â±flar: {class_names}")
print(f"EÃ„Å¸itim Verisi: {dataset_sizes['train']}")
print(f"Validasyon Verisi: {dataset_sizes['val']}")

In [ ]:
def create_model(model_name: str, num_classes: int, dropout_rate: float, device: str):
    print(f"Model indiriliyor: {model_name}...")

    # pretrained=True ile ImageNet aÃ„Å¸Ã„Â±rlÃ„Â±klarÃ„Â±nÃ„Â± alÃ„Â±yoruz
    try:
        model = timm.create_model(model_name,
                                  pretrained=True,
                                  num_classes=num_classes,
                                  drop_rate=dropout_rate)
    except Exception as e:
        print(f"Hata: Model '{model_name}' yÃƒÂ¼klenemedi. Kontrol edin. Hata: {e}")
        return None

    model = model.to(device)
    return model
model = create_model(MODEL_NAME, NUM_CLASSES, DROPOUT_RATE, DEVICE)
criterion = nn.CrossEntropyLoss()
weight_decay = 1e-4 if 'efficientnet' in MODEL_NAME else 5e-2 # CvT iÃƒÂ§in 5e-2 daha agresif.

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=weight_decay)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print("Model GPU'ya yÃƒÂ¼klendi ve eÃ„Å¸itime hazÃ„Â±r.")

In [ ]:
def train_model(model, criterion, optimizer, scheduler, num_epochs):
    since = time.time()
    best_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}


    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            # Batch DÃƒÂ¶ngÃƒÂ¼sÃƒÂ¼
            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(DEVICE)
                labels = labels.to(DEVICE)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    # Tekrar kontrol: outputs, (N, C) boyutunda olmalÃ„Â±dÃ„Â±r.
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            # Epoch Sonucu Hesaplama
            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # GeÃƒÂ§miÃ…Å¸i Kaydet
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            # --- SCHEDULER VE MODEL KAYDI DÃƒÅ“ZENLEMELERÃ„Â° ---

            if phase == 'train':
                # CosineAnnealingLR: Her eÃ„Å¸itim epoch'undan sonra adÃ„Â±m atmalÃ„Â±dÃ„Â±r.
                # (ReduceLROnPlateau gibi val kaybÃ„Â±na baÃ„Å¸lÃ„Â± deÃ„Å¸ildir)
                # Sadece eÃ„Å¸itim fazÃ„Â±nda gÃƒÂ¼ncellenir.
                if isinstance(scheduler, (torch.optim.lr_scheduler.CosineAnnealingLR, torch.optim.lr_scheduler.OneCycleLR)):
                    scheduler.step()
                    # GÃƒÂ¼ncel LR'Ã„Â± gÃƒÂ¶rmek faydalÃ„Â± olabilir
                    current_lr = optimizer.param_groups[0]['lr']
                    print(f"Current LR: {current_lr:.6f}")

            if phase == 'val':
                # ReduceLROnPlateau kullanÃ„Â±yorsanÃ„Â±z, burayÃ„Â± tekrar aÃƒÂ§Ã„Â±n:
                if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                    scheduler.step(epoch_loss)

                # En iyi modeli kaydet (Validasyon baÃ…Å¸arÃ„Â±sÃ„Â±na gÃƒÂ¶re)
                if epoch_acc > best_acc:
                    best_acc = epoch_acc
                    save_path = os.path.join(OUTPUT_DIR, 'best_model.pth')
                    torch.save(model.state_dict(), save_path)
                    print(f"En Ã„Â°yi Model! ({best_acc:.4f}) -> Kaydedildi ve Yolu: {save_path}")

    time_elapsed = time.time() - since
    print(f'\nEÃ„Å¸itim TamamlandÃ„Â±: {time_elapsed // 60:.0f}dk {time_elapsed % 60:.0f}sn')
    print(f'En Ã„Â°yi Validasyon DoÃ„Å¸ruluÃ„Å¸u: {best_acc:.4f}')

    # En iyi aÃ„Å¸Ã„Â±rlÃ„Â±klarÃ„Â± geri yÃƒÂ¼kle
    model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, 'best_model.pth')))
    return model, history